# LinguaPinas: record the spirits' voices with SapinSapin's native-speaker models

This notebook records every Philippine-language word, sentence and Filipino line of the Guiding Spirits
with [SapinSapin](https://github.com/sapinsapin/halohalo)'s Orpheus 3B voice models, one per language,
trained on recordings of real native speakers.

1. **Runtime → Change runtime type → T4 GPU** (free), then run the cells from top to bottom.
2. The models (about 7 GB) are downloaded from Hugging Face the first time.
3. Recording all 242 clips with `--check` takes roughly 30–60 minutes on a T4.

The SapinSapin models are for research and non-commercial use (CC-BY-NC 4.0).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the game
If the repository is private, paste a GitHub token (with access to it) when asked; otherwise just press Enter.

In [ ]:
import getpass, os
token = getpass.getpass("GitHub token (Enter if the repo is public): ").strip()
url = "https://github.com/norbencent/Linguapinas_1"
if token:
    url = url.replace("https://", f"https://{token}@")
if not os.path.exists("linguapinas"):
    !git clone --depth 1 {url} linguapinas
%cd linguapinas

In [ ]:
!pip install -q transformers peft accelerate snac soundfile librosa kokoro-onnx
!ffmpeg -version | head -1

## 2. Listen to the speakers and choose each spirit's voice
This records a greeting and a lesson sentence with every speaker listed in `database/spirits.json`
(`voice` → `orpheus`). Maelor should sound like a calm older man; Marea and Lanora like women.

In [ ]:
!python tools/voice/make_spirit_voices_orpheus.py --audition

In [ ]:
import glob
from IPython.display import Audio, display
for path in sorted(glob.glob("tools/voice/samples/orpheus/*.ogg")):
    print(path.split("/")[-1])
    display(Audio(path))

Put the speaker you like **first** for each spirit and language (leave a language out to keep the order as it is),
then run the cell.

In [ ]:
import json
choice = {
    # "luzon":    {"fil": "FIL_0008", "ilo": "ILO_1755", "pam": "PAM_0402", "bcl": "BIK_0982"},
    # "visayas":  {"fil": "FIL_0003", "ceb": "CEB_0312", "hil": "HIL_0624", "war": "WAR_1047"},
    # "mindanao": {"fil": "FIL_0000", "ceb": "CEB_0314", "tsg": "TSG_1450"},
}
path = "database/spirits.json"
data = json.load(open(path, encoding="utf-8"))
for region, langs in choice.items():
    for lang, speaker in langs.items():
        ids = data["spirits"][region]["voice"]["orpheus"][lang]
        data["spirits"][region]["voice"]["orpheus"][lang] = [speaker] + [s for s in ids if s != speaker]
open(path, "w", encoding="utf-8").write(json.dumps(data, ensure_ascii=False, indent=1) + "\n")
print({r: {l: v[0] for l, v in s["voice"]["orpheus"].items()} for r, s in data["spirits"].items()})

## 3. Record all the clips
`--check` transcribes every take with SapinSapin's speech recognition and keeps the clearest of up to 3 takes.
Clips that are still hard to understand are listed in `tools/voice/orpheus_report.json`.

In [ ]:
!python tools/voice/make_spirit_voices_orpheus.py --check

In [ ]:
import json
report = json.load(open("tools/voice/orpheus_report.json", encoding="utf-8"))
print(len(report), "clips to listen to again")
for r in report[:20]:
    print(r)
    display(Audio(r["file"]))

## 4. Send the new voices back
**Option A (easiest with a token):** push them to a new branch, then tell Claude "merge the voices-orpheus branch".

In [ ]:
!git config user.email "you@example.com" && git config user.name "LinguaPinas"
!git checkout -b voices-orpheus
!git add assets/voice database/spirits.json
!git commit -q -m "Record the Philippine-language voices with SapinSapin Orpheus models"
!git push -q origin voices-orpheus && echo "Pushed to the voices-orpheus branch."

**Option B:** download a zip and copy its folders over the same folders in your project.

In [ ]:
!zip -qr linguapinas_voices.zip assets/voice database/spirits.json
from google.colab import files
files.download("linguapinas_voices.zip")